In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "volter2015exploitation")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Voelter_2015_exp_1_door_AnimBehav_FRDS .csv")
complete_path_2 = os.path.join(original_data_pathway, "Voelter_2015_exp_1_platform_AnimBehav_FRDS .csv")
complete_path_3 = os.path.join(original_data_pathway, "Voelter_2015_exp_2 door_AnimBehav_FRDS .csv")
complete_path_4 = os.path.join(original_data_pathway, "Voelter_2015_exp_2 platform_AnimBehav_FRDS .csv")
complete_path_5 = os.path.join(original_data_pathway, "Voelter_2015_exp_3 door_AnimBehav_FRDS .csv")
complete_path_6 = os.path.join(original_data_pathway, "Voelter_2015_exp_3 platform_AnimBehav_FRDS .csv")
complete_path_7 = os.path.join(original_data_pathway, "volter2015exploitation_exp4_standardized_author_emailed.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)
df3 = pd.read_csv(complete_path_3)
df4 = pd.read_csv(complete_path_4)
df5 = pd.read_csv(complete_path_5)
df6 = pd.read_csv(complete_path_6)
df7 = pd.read_csv(complete_path_7)

df7.columns

df7 = df7[['participant',  'participant_2', 
       'session', 'trial_number', 'condition', 'baiting_room', 'tool_given',
       'tool_inserted_by', '2_reward_retrieved_by', '4_reward_retrieved_by',
       '2_reward_eaten_by', '4_reward_eaten_by', 'tool_transfer_by_mother',
       'social_tool', 'location_of_infant_when_mother_received_tool']]
df7.rename(columns={"participant": "subject"}, inplace=True)
df7.rename(columns={"participant_2": "child"}, inplace=True)

In [3]:
experiment_import = [[df1, '1','door'],
                    [df2, '1', 'platform'],
                    [df3, '2','door'],
                    [df4, '2','platform'],
                    [df5, '3', 'door'],
                    [df6, '3','platform'],
                    [df7, '4', 'tube']]
for x,y,k in experiment_import:
    x['experiment']=y
    x['experiment_name']=k

In [4]:
data_frames=[df1, df2, df3, df4, df5, df6, df7]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"subject": "ape",
        "child":"ape_2",
        "species":"species_original",
        "tool given":"tool_given",
        "tool transfer by mother":"tool_transfer_by_mother",
        "location of infant when mother received tool":"location_of_infant_when_mother_received_tool"}, inplace=True)
    x['study_id']="volter2015exploitation"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

fulldf = fulldf.assign(role='mother')
fulldf = fulldf.assign(role_2='child')

In [5]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)
    fulldf['ape_2'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left') 

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
fulldf= fulldf.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')


fulldf['dyad']=fulldf.ape.str.cat(fulldf.ape_2, sep='_')

In [6]:
# fulldf.columns
fulldf.rename(columns={"ape": "participant", "ape_2":"participant_2",
                       "trial id":"trial_id"}, inplace=True)

In [7]:
fulldf=fulldf[['study_id','experiment', 'experiment_name','participant','sex','role', 'participant_2', 'sex_2', 'role_2','species','dyad',
        'session', 'trial','trial_id','trial_number', 'condition',
       'number_reward', 'reward_retrieved_by', 'reward_eaten_by',
       'social_tool_use', 'hold', 'pull', 'recruit', 'push', 
        'guide', 'tool_retrieved_by',  'type',
       'first_choice_type', 'first_choice_direction', 'retrieved_by',
       'second_choice', 'food_eaten', 'tool_obtained', 'food_obtained',
       'distractor_obtained', 'mother_eats', 'second_reward_eaten_by',
       'baiting_room',  'tool_given', 'tool_inserted_by',
       '2_reward_retrieved_by', '4_reward_retrieved_by', '2_reward_eaten_by',
       '4_reward_eaten_by', 'tool_transfer_by_mother', 'social_tool',
       'location_of_infant_when_mother_received_tool']]


In [8]:
for index in range(1,5):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'volter2015exploitation_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'volter2015exploitation_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)